# BERT Fine-tuning, Training & Evaluation

This notebook covers:
1. Data loading and splitting (train/val/test)
2. BERT model fine-tuning with TensorFlow
3. Training with callbacks (early stopping, checkpoints)
4. Evaluation on test set
5. Visualization of results

## 1. Setup & Imports

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split
metric = evaluate.load("f1")
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve repo root
from pathlib import Path

def find_repo_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / "README.md").exists() or (candidate / "src").exists() or (candidate / ".git").exists():
            return candidate
    return start

BASE_DIR = find_repo_root(Path.cwd())
RESULTS_DIR = BASE_DIR / "results"
MODELS_DIR = RESULTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {BASE_DIR}")
print(f"Models dir: {MODELS_DIR}")

Repo root: c:\Life\FCAI\Specialization - AI\Third_Year\2nd Semester\3 - Supervised Learning\7- Projects\nlp-fake-news-detector-transformers
Models dir: c:\Life\FCAI\Specialization - AI\Third_Year\2nd Semester\3 - Supervised Learning\7- Projects\nlp-fake-news-detector-transformers\results\models


## 2. Load Data & Create Train/Val/Test Split

In [2]:
# Load processed data from repo-relative path
df = pd.read_csv(BASE_DIR / "data" / "processed" / "processed.csv")
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:\n{df['target'].value_counts()}")
print(f"\nData info:")
df.head()

Total samples: 1482190

Class distribution:
target
0    749664
1    732526
Name: count, dtype: int64

Data info:


,cleaned_text,target
0,awww thats bummer shoulda got david carr third...,0
1,upset cant update facebook texting might cry r...,0
2,dived many times ball managed save rest go bounds,0
3,whole body feels itchy like fire,0
4,not behaving im mad cant see,0


In [3]:
# Extract texts and labels
texts = df['cleaned_text'].astype(str).values
labels = df['target'].values

# First split: 80% train+val, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Second split: 75% train (60% of total), 25% val (20% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Train set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTrain class distribution: {np.bincount(y_train)}")
print(f"Val class distribution: {np.bincount(y_val)}")
print(f"Test class distribution: {np.bincount(y_test)}")

Train set size: 889314
Validation set size: 296438
Test set size: 296438

Train class distribution: [449798 439516]
Val class distribution: [149933 146505]
Test class distribution: [149933 146505]


## 3. Prepare Datasets using BertDataPreparer

In [4]:
# Tokenizer and dataset preparation (PyTorch)
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LENGTH = 128
BATCH_SIZE = 16

# Extract texts and labels
texts = df['cleaned_text'].astype(str).tolist()
labels = df['target'].astype(int).tolist()

# Train/val/test split
X_train_val, X_test, y_train_val, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

print(f"Train set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

# Create HuggingFace datasets
train_ds = Dataset.from_dict({'text': X_train, 'label': y_train})
val_ds = Dataset.from_dict({'text': X_val, 'label': y_val})
test_ds = Dataset.from_dict({'text': X_test, 'label': y_test})

def tokenize_batch(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

# Set format for PyTorch
train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print('Datasets tokenized and formatted for PyTorch')

# Inspect a batch
sample = train_ds[0]
print({k: v.shape if hasattr(v, 'shape') else type(v) for k, v in sample.items()})

Train set size: 889314
Validation set size: 296438
Test set size: 296438


Map:   0%|          | 0/889314 [00:00<?, ? examples/s]

Map:   0%|          | 0/296438 [00:00<?, ? examples/s]

Map:   0%|          | 0/296438 [00:00<?, ? examples/s]

Datasets tokenized and formatted for PyTorch
{'label': torch.Size([]), 'input_ids': torch.Size([128]), 'attention_mask': torch.Size([128])}


## 4. Build BERT Fine-tuning Model

In [5]:
from transformers import AutoModelForSequenceClassification

def build_bert_model(model_name='bert-base-uncased', num_labels=2):
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    return model

model = build_bert_model(model_name=model_name, num_labels=2)
print('Model built successfully')
print(model.config)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model built successfully
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.8.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



## 5. Compile Model

In [6]:
# Note: PyTorch Trainer handles optimization internally via TrainingArguments
# No manual optimizer or compile step needed
print("Model ready for training via Trainer")

Model ready for training via Trainer


## 6. Define Callbacks

In [8]:
from transformers import TrainingArguments, Trainer
import numpy as np

# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / 'bert'),
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Trainer configured')

Trainer configured


## 7. Train Model

In [ ]:
# Train model with HuggingFace Trainer (PyTorch)
train_result = trainer.train()

print('\nTraining completed!')
print(train_result)

## 8. Save Training History

In [ ]:
import json

# Save training history from Trainer metrics
history_dict = {
    'train_loss': train_result.training_loss if hasattr(train_result, 'training_loss') else None,
    'eval_loss': train_result.best_metric if hasattr(train_result, 'best_metric') else None,
}

history_file = MODELS_DIR / 'bert' / 'training_history.json'
with open(history_file, 'w') as f:
    json.dump(history_dict, f, indent=4)

print(f"Training history saved to {history_file}")

## 9. Evaluate on Test Set

In [ ]:
# Get predictions on test set using Trainer
test_predictions = trainer.predict(test_ds)
y_pred_probs = test_predictions.predictions
y_pred = np.argmax(y_pred_probs, axis=1)

# Convert y_test to numpy array for sklearn
y_test_np = np.array(y_test)

# Calculate metrics
accuracy = accuracy_score(y_test_np, y_pred)
precision = precision_score(y_test_np, y_pred, zero_division=0)
recall = recall_score(y_test_np, y_pred, zero_division=0)
f1 = f1_score(y_test_np, y_pred, zero_division=0)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test_np, y_pred, target_names=['Not Fake', 'Fake']))

## 10. Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_np, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Fake', 'Fake'],
            yticklabels=['Not Fake', 'Fake'])
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
cm_file = MODELS_DIR / 'bert' / 'confusion_matrix.png'
cm_file.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(cm_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"\nConfusion Matrix:\n{cm}")
print(f"Confusion matrix saved to {cm_file}")

## 11. Training Curves

In [ ]:
# Load training logs from Trainer output directory
import json
from pathlib import Path

trainer_log_dir = MODELS_DIR / 'bert'
log_files = list(trainer_log_dir.glob('**/trainer_state.json'))

if log_files:
    with open(log_files[0], 'r') as f:
        trainer_state = json.load(f)
    
    train_loss = [log.get('loss', None) for log in trainer_state.get('log_history', []) if 'loss' in log and 'eval' not in log]
    val_loss = [log.get('eval_loss', None) for log in trainer_state.get('log_history', []) if 'eval_loss' in log]
    
    fig, axes = plt.subplots(1, 1, figsize=(10, 5))
    
    if train_loss:
        axes.plot(range(len(train_loss)), train_loss, label='Train Loss', marker='o')
    if val_loss:
        axes.plot(range(len(val_loss)), val_loss, label='Eval Loss', marker='s')
    
    axes.set_xlabel('Step')
    axes.set_ylabel('Loss')
    axes.set_title('Model Training Loss')
    axes.legend()
    axes.grid(True, alpha=0.3)
    
    plt.tight_layout()
    curves_file = MODELS_DIR / 'bert' / 'training_curves.png'
    plt.savefig(curves_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Training curves saved to {curves_file}")
else:
    print("No training state file found. Skipping training curves plot.")

## 12. Save Evaluation Results

In [ ]:
# Save evaluation results
results = {
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1_score': float(f1),
    'confusion_matrix': cm.tolist(),
    'model_name': 'BERT Fine-tuned (PyTorch)',
    'num_train_epochs': 3,
    'batch_size': BATCH_SIZE,
    'max_length': MAX_LENGTH,
}

results_file = MODELS_DIR / 'bert' / 'evaluation_results.json'
results_file.parent.mkdir(parents=True, exist_ok=True)

with open(results_file, 'w') as f:
    json.dump(results, f, indent=4)

print(f"\nEvaluation results saved to {results_file}")

## 13. Save Full Model

In [ ]:
# Save the model and tokenizer using HuggingFace format
model_save_dir = MODELS_DIR / 'bert' / 'final_model'
model_save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(model_save_dir))
tokenizer.save_pretrained(str(model_save_dir))

print(f"Model saved to {model_save_dir}")
print(f"Tokenizer saved to {model_save_dir}")

## 14. Model Predictions on Test Data

In [ ]:
# Show predictions on actual test data
print("\n" + "=" * 80)
print("SAMPLE PREDICTIONS FROM TEST SET")
print("=" * 80)

# Display first 10 test samples with predictions
num_samples = min(10, len(X_test))

for idx in range(num_samples):
    text = X_test[idx]
    true_label = "Fake" if y_test_np[idx] == 1 else "Not Fake"
    pred_label = "Fake" if y_pred[idx] == 1 else "Not Fake"
    confidence = np.max(y_pred_probs[idx]) * 100
    match = "✓" if y_pred[idx] == y_test_np[idx] else "✗"
    
    print(f"\n[Sample {idx+1}] {match}")
    print(f"Text: {text[:80]}...")
    print(f"True Label:      {true_label}")
    print(f"Predicted Label: {pred_label} ({confidence:.2f}% confidence)")

print("\n" + "=" * 80)
print(f"\nTotal Test Samples: {len(X_test)}")
print(f"Correct Predictions: {(y_pred == y_test_np).sum()} / {len(y_test_np)}")